# Gymnasium: BipedalWalker-v3

Our objective is to train an agent to navigate the BipedalWalker environment using Reinforcement Learning. Before implementing complex algorithms or aiming for advanced maneuvers (like doing a flip), we need to understand the environment's dynamics.

## Environment Overview
`BipedalWalker-v3` is a 2D physics simulation environment from the Gymnasium Box2D environments. The goal is to make a bipedal robot walk to the right end of the terrain. In the `hardcore=True` version, the terrain is not flat; it includes obstacles such as ladders, stumps, and pitfalls.

### Action Space
The action space is a continuous `Box(-1.0, 1.0, (4,), float32)`. The agent controls the robot by applying torques to its four main joints. The four values in the action array represent:
1. Hip 1 (Torque / Speed)
2. Knee 1 (Torque / Speed)
3. Hip 2 (Torque / Speed)
4. Knee 2 (Torque / Speed)

All action values must be within the `[-1.0, 1.0]` range.

### Reward System
The agent receives rewards based on its forward progress and energy efficiency. According to the official documentation, the reward is calculated as follows:
* **Forward Movement:** The agent is rewarded for moving forward (to the right). Reaching the end of the terrain yields a total of over 300 points.
* **Falling Penalty:** If the robot's main body (hull) touches the ground, it falls. This results in a heavy penalty of **-100** points, and the episode terminates immediately.
* **Motor Torque Penalty:** To encourage efficient, natural walking rather than chaotic flailing, applying motor torque costs a small negative reward.
### Set up the Environment

In [11]:
import gymnasium as gym
env = gym.make("BipedalWalker-v3", hardcore=False, render_mode="human") # Human we can see the environment

## Baseline: Random Actions

To establish a baseline and visualize how an untrained agent interacts with the physics engine, we will run a single episode using a random policy. The agent will sample actions uniformly from the action space until the episode ends.

An episode ends if:
- `terminated` is True (the agent falls or reaches the goal).
- `truncated` is True (the agent runs out of time/steps).
- Past 20 seconds of simulation time.

In [21]:
import time

env = gym.make("BipedalWalker-v3", hardcore=False, render_mode="human") 
limit_time = 10  # seconds
start_time = time.time()
obs, info = env.reset()

terminated = False
truncated = False
total_reward = 0.0
step_count = 0

# Loop until the agent finishes or fails
while not (terminated or truncated) and (time.time() - start_time < limit_time):
    # Sample a random continuous action within [-1, 1] for the 4 joints
    action = env.action_space.sample() 
    
    # Step the environment forward
    obs, reward, terminated, truncated, info = env.step(action)
    
    total_reward += reward
    step_count += 1

# Close the rendering window
env.close()

print(f"Episode finished after {step_count} steps.")
print(f"Total Reward with random policy: {total_reward:.2f}")

Episode finished after 52 steps.
Total Reward with random policy: -105.10


### Changing the Environment
Instead of training the agent only to walk, we can reshape the task so it learns to perform a flip.
The main idea is to change the reward and encourage trunk rotation, airtime, and landing control.


Changes in the robot:
- Track the robot body's orientation.
- Reward angular velocity and rotation progress.
- Give a large bonus when the agent completes a full rotation.
- Reduce the fall penalty so the agent is willing to take risks.
- Penalize forward movement less, so the policy focuses on flipping rather than walking.

In [ ]:
import gymnasium as gym
import numpy as np

class BipedalFlipperWrapper(gym.Wrapper):
    def __init__(self, env, max_stagnation_steps=250):
        super().__init__(env)
        self.cumulative_angle = 0.0
        self.prev_angle = 0.0
        self.flip_completed = False
        self.max_stagnation_steps = max_stagnation_steps
        self.step_counter = 0
        self.last_x = 0.0
        self.last_progress_step = 0

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.cumulative_angle = 0.0
        self.prev_angle = obs[0]  # hull angle
        self.flip_completed = False
        self.step_counter = 0
        self.last_progress_step = 0
        try:
            self.last_x = self.env.unwrapped.hull.position.x
        except AttributeError:
            self.last_x = 0.0
        return obs, info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        self.step_counter += 1
        try:
            current_x = self.env.unwrapped.hull.position.x
            if current_x > self.last_x + 1.0:
                self.last_x = current_x
                self.last_progress_step = self.step_counter
        except AttributeError:
            pass
        stagnation_duration = self.step_counter - self.last_progress_step
        if stagnation_duration >= self.max_stagnation_steps:
            terminated = True
        current_angle = obs[0]
        delta_angle = current_angle - self.prev_angle
        if delta_angle > np.pi:
            delta_angle -= 2 * np.pi
        elif delta_angle < -np.pi:
            delta_angle += 2 * np.pi
        self.cumulative_angle += delta_angle
        self.prev_angle = current_angle
        # Reward rotation more aggressively so the policy prefers attempting flips.
        custom_reward = -delta_angle * 20.0
        if self.cumulative_angle <= -2 * np.pi and not self.flip_completed:
            custom_reward += 1000.0
            self.flip_completed = True
            terminated = True
        if reward == -100:
            if self.flip_completed:
                custom_reward = 0
            else:
                custom_reward -= 1
        custom_reward -= sum(abs(action)) * 0.01
        info.setdefault('flip_completed', self.flip_completed)
        info.setdefault('cumulative_angle', self.cumulative_angle)
        return obs, custom_reward, terminated, truncated, info

## Integrating Stable Baselines 3

Using Stable Baselines 3, we can implement a Proximal Policy Optimization (PPO) agent to learn how to flip in the BipedalWalker environment.

In [ ]:
import os
import torch
import gymnasium as gym
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.callbacks import BaseCallback

class RenderCallback(BaseCallback):
    def __init__(self, render_freq: int, verbose=0):
        super().__init__(verbose)
        self.render_freq = render_freq
        self.render_env = None

    def _on_step(self) -> bool:
        if self.num_timesteps > 0 and self.num_timesteps % self.render_freq == 0:
            print(f"\n--- [Step {self.num_timesteps}] Demonstration ---")
            if self.render_env is None:
                e = gym.make("BipedalWalker-v3", hardcore=False, render_mode="human")
                self.render_env = BipedalFlipperWrapper(e)
            obs, _ = self.render_env.reset()
            done = False
            while not done:
                action, _ = self.model.predict(obs, deterministic=True)
                obs, _, terminated, truncated, _ = self.render_env.step(action)
                done = terminated or truncated
            print("--- End demonstration. Resuming training ---\n")
        return True


def make_env():
    def _init():
        e = gym.make("BipedalWalker-v3", hardcore=False)
        return BipedalFlipperWrapper(e)
    return _init

num_envs = 32  # Ajuste livremente este valor para escalar as simulações
print(f"Creating {num_envs} parallel environments...")
vec_env = SubprocVecEnv([make_env() for _ in range(num_envs)])
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

model_path = "sac_bipedal_flipper.zip"
if os.path.exists(model_path):
    print(f"Loading existing model from {model_path}...")
    model = SAC.load(model_path, env=vec_env, device=device)
else:
    print("Initializing new SAC model on device", device)
    model = SAC("MlpPolicy", vec_env, verbose=1, device=device)

total_steps = 100000
print(f"Starting training for {total_steps} timesteps...")

# Toggle periodic human-render demonstrations:
use_render_demos = False  # Set to True to enable demonstrations
render_freq = 2000

eval_callback = RenderCallback(render_freq=render_freq) if use_render_demos else None

model.learn(total_timesteps=total_steps, callback=eval_callback)
print("Training finished! Saving model...")
model.save("sac_bipedal_flipper")
vec_env.close()

Creating 32 parallel environments...
Using device: cpu
Loading existing model from sac_bipedal_flipper.zip...
Starting training for 100000 timesteps...
---------------------------------
| time/              |          |
|    episodes        | 4        |
|    fps             | 1806     |
|    time_elapsed    | 1        |
|    total_timesteps | 3520     |
| train/             |          |
|    actor_loss      | -22.2    |
|    critic_loss     | 1.48     |
|    ent_coef        | 0.0608   |
|    ent_coef_loss   | -5.67    |
|    learning_rate   | 0.0003   |
|    n_updates       | 10692    |
---------------------------------
---------------------------------
| time/              |          |
|    episodes        | 8        |
|    fps             | 1664     |
|    time_elapsed    | 4        |
|    total_timesteps | 6912     |
| train/             |          |
|    actor_loss      | -21.1    |
|    critic_loss     | 0.785    |
|    ent_coef        | 0.0592   |
|    ent_coef_loss   | -4.94    

### Testing Model


In [14]:
import time
import os
import torch
import gymnasium as gym
from stable_baselines3 import SAC

model_path = "sac_bipedal_flipper.zip"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Create a single human-render environment and wrap it
env = gym.make("BipedalWalker-v3", hardcore=False, render_mode="human")
env = BipedalFlipperWrapper(env)

if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model not found: {model_path}. Train and save the model before running tests.")

print(f"Loading model from {model_path}...")
model = SAC.load(model_path, device=device)

num_episodes = 5
max_seconds = 10

for ep in range(1, num_episodes + 1):
    obs, info = env.reset()
    start_time = time.time()
    done = False
    total_reward = 0.0
    step_count = 0
    last_reward = 0.0

    print(f"\n=== Test Episode {ep} ===")
    while not done and (time.time() - start_time) < max_seconds:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        total_reward += reward
        last_reward = reward
        step_count += 1

    # Infer whether the agent fell: original env gives reward -100 on fall
    fell = info.get("fell", False)
    flip_completed = info.get("flip_completed", False)

    print(f"Episode {ep} — steps: {step_count}, total_reward: {total_reward:.2f}, fell: {fell}, flip_completed: {flip_completed}")

    # Small pause between episodes so you can see the results
    time.sleep(0.5)

env.close()
print("All tests finished.")

Using device: cpu
Loading model from sac_bipedal_flipper.zip...

=== Test Episode 1 ===
Episode 1 — steps: 218, total_reward: 22.54, fell: False, flip_completed: False

=== Test Episode 2 ===
Episode 2 — steps: 250, total_reward: 3.33, fell: False, flip_completed: False

=== Test Episode 3 ===
Episode 3 — steps: 250, total_reward: 13.88, fell: False, flip_completed: False

=== Test Episode 4 ===
Episode 4 — steps: 261, total_reward: 22.01, fell: False, flip_completed: False

=== Test Episode 5 ===
Episode 5 — steps: 250, total_reward: 3.52, fell: False, flip_completed: False
All tests finished.
